# RAG LLM LawBot

## Install relevant packages

In [1]:
%%capture

!pip install unsloth
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -U sentence-transformers
!pip install -q langchain==0.1.20
!pip install -q langchain-community==0.0.38
!pip install -q chromadb==0.4.24
!pip install -q gradio==4.36.1
!pip install -q pymupdf==1.23.8
!pip install -q sentence-transformers==2.2.2
!pip install youtube_dl
!pip install whisper
!pip install -U huggingface_hub
!pip install -U sentence-transformers
!pip install rank-bm25
!pip install rouge-score
!pip install bert-score
!pip install sentence-transformers
!pip install Sastrawi

## Import all relevant packages throughout this walkthrough

In [2]:
# Modules for fine-tuning
from unsloth import FastLanguageModel
import torch # Import PyTorch
from trl import SFTTrainer # Trainer for supervised fine-tuning (SFT)
from unsloth import is_bfloat16_supported # Checks if the hardware supports bfloat16 precision
# Hugging Face modules
from huggingface_hub import login # Lets you login to API
from transformers import TrainingArguments # Defines training hyperparameters
from datasets import load_dataset # Lets you load fine-tuning datasets
# Import weights and biases
import wandb
# Import kaggle secrets
from kaggle_secrets import UserSecretsClient

import pandas as pd
import re
import string
from sklearn.metrics import precision_score, recall_score
import random
from typing import List, Dict, Tuple

# Import necessary packages
import gradio as gr
import torch
import re
import os
from pathlib import Path
import warnings

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma

# Embedding generation using HuggingFace embeddings instead of Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings

# Import kaggle secrets for token management
from kaggle_secrets import UserSecretsClient

from transformers import pipeline

## Import all relevant packages
import torch
import re
import os
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import youtube_dl
import whisper

# Hugging Face modules
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Unsloth and model imports
from unsloth import FastLanguageModel
from transformers import pipeline

# Document processing and retrieval
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.schema import Document

# Alternative embedding using transformers directly
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

from IPython.display import FileLink, display

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-06-21 12:39:34.648996: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750509574.854443      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750509574.913316      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# warnings.filterwarnings("ignore", message="Unexpected keyword arguments")

## Create API keys and login to Hugging Face and Weights and Biases

In [4]:
user_secrets = UserSecretsClient()
hugging_face_token = user_secrets.get_secret("HF_TOKEN_DEEPSEEK")
wnb_token = user_secrets.get_secret("wnb_token")

login(hugging_face_token)

wandb.login(key=wnb_token)
run = wandb.init(
    project='LawBot LLM Judge RAG', 
    job_type="training", 
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: naufalkr394 (naufalkr394-institut-teknologi-sepuluh-nopember) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load model adapter menggunakan Unsloth
model_lora_lawbot, tokenizer= FastLanguageModel.from_pretrained(
    model_name="luckysantoso/adapter-sealion-lawbot-v2",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    trust_remote_code=True,
    token=hugging_face_token,
)

FastLanguageModel.for_inference(model_lora_lawbot)


# from transformers import AutoTokenizer
# from unsloth import FastLanguageModel

# MODEL_DIR = "/kaggle/input/model-lora-qwen/model_lora_lawbot_final"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

# model_lora_lawbot, _ = FastLanguageModel.from_pretrained(
#     model_name=MODEL_DIR,
#     max_seq_length=2048,
#     dtype=None,            
#     load_in_4bit=True,     
# )

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.6.3: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 6.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

aisingapore/Llama-SEA-LION-v3-8B-IT does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


adapter_model.safetensors:   0%|          | 0.00/37.8M [00:00<?, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.6.3 patched 32 layers with 32 QKV layers, 0 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

# **RAG PEFT Implementation**

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
from rouge_score import rouge_scorer
from bert_score import score
import json
from tqdm import tqdm
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
import numpy as np
from sentence_transformers import CrossEncoder

In [7]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

In [8]:
# Text preprocessing with Indonesian-specific handling
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    text = re.sub(r'[^\w\s\-.,;:!?()\[\]/]', '', text)
    
    legal_replacements = {
        'Undang-Undang': 'UU',
        'undang-undang': 'UU',
        'Tentara Nasional Indonesia': 'TNI',
        'Operasi Militer Selain Perang': 'OMSP',
        'Angkatan Bersenjata Republik Indonesia': 'ABRI'
    }
    
    for old, new in legal_replacements.items():
        text = text.replace(old, new)
    
    return text

In [9]:
def load_legal_json(json_path):
    documents = []
    
    try:
        print(f"Loading legal JSON from: {json_path}")
        with open(json_path, 'r', encoding='utf-8') as f:
            legal_data = json.load(f)
        
        for item in legal_data:
            pasal_num = item.get('pasal', 'Unknown')
            isi = item.get('isi', '')
            penjelasan = item.get('penjelasan', '')
            isi_perubahan = item.get('isiPerubahan', '')
            penjelasan_perubahan = item.get('penjelasanPerubahan', '')
            is_changed = item.get('isChanged', False)
            
            content_parts = []
            content_parts.append(f"Pasal {pasal_num}")
            
            if isi:
                content_parts.append(f"Isi: {isi}")
            
            if penjelasan and penjelasan.strip() != "Cukup jelas.":
                content_parts.append(f"Penjelasan: {penjelasan}")
            
            if is_changed and isi_perubahan:
                content_parts.append(f"Perubahan: {isi_perubahan}")
                
                if penjelasan_perubahan:
                    content_parts.append(f"Penjelasan Perubahan: {penjelasan_perubahan}")
            
            full_content = "\n\n".join(content_parts)
            full_content = preprocess_text(full_content)
            
            if len(full_content) > 50:
                documents.append(Document(
                    page_content=full_content,
                    metadata={
                        "source": json_path,
                        "pasal": pasal_num,
                        "type": "legal_article",
                        "is_changed": is_changed,
                        "content_length": len(full_content),
                        "doc_type": "undang_undang"
                    }
                ))
        
        print(f"Loaded {len(documents)} legal articles from JSON")
        
    except Exception as e:
        print(f"Error loading legal JSON: {e}")
    
    return documents

In [10]:
def load_pdf_documents(pdf_paths):
    documents = []
    
    for pdf_path in pdf_paths:
        try:
            print(f"Loading PDF: {pdf_path}")
            loader = PyMuPDFLoader(pdf_path)
            pages = loader.load()
            
            doc_type = "unknown"
            if "naskah-akademik" in pdf_path.lower():
                doc_type = "academic_draft"
            elif "petisi" in pdf_path.lower():
                doc_type = "petition"
            elif "permohonan" in pdf_path.lower():
                doc_type = "legal_request"
            elif "wiraedsus" in pdf_path.lower():
                doc_type = "analysis_report"
            
            for page in pages:
                content = preprocess_text(page.page_content)
                if len(content) > 100:
                    documents.append(Document(
                        page_content=content,
                        metadata={
                            "source": pdf_path,
                            "page": page.metadata.get("page", 0),
                            "type": "pdf",
                            "doc_type": doc_type,
                            "content_length": len(content)
                        }
                    ))
            print(f"Loaded {len([d for d in documents if d.metadata['source'] == pdf_path])} pages from {pdf_path}")
            
        except Exception as e:
            print(f"Error loading {pdf_path}: {e}")
    
    return documents

In [11]:
def load_web_articles(csv_path):
    documents = []
    
    try:
        print(f"Loading web articles from: {csv_path}")
        df = pd.read_csv(csv_path)
        
        for _, row in df.iterrows():
            title = str(row.get('Title', ''))
            content = str(row.get('Content', ''))
            date = str(row.get('Date', ''))
            url = str(row.get('URL', ''))
            
            full_content = f"Judul: {title}\n\nIsi: {content}"
            full_content = preprocess_text(full_content)
            
            if len(full_content) > 100:
                documents.append(Document(
                    page_content=full_content,
                    metadata={
                        "source": url,
                        "title": title,
                        "date": date,
                        "type": "web_article",
                        "content_length": len(full_content)
                    }
                ))
        
        print(f"Loaded {len(documents)} web articles")
        
    except Exception as e:
        print(f"Error loading web articles: {e}")
    
    return documents

In [12]:
PDF_SOURCES = [
    "/kaggle/input/rag-pdf-v2/Permohonan_4288_8190_Permohonan_redact.pdf",
    "/kaggle/input/rag-pdf-v2/Petisi-Revisi-UU-TNI-.pdf", 
    "/kaggle/input/rag-pdf-v2/naskah-akademik.pdf",
    "/kaggle/input/rag-pdf-v2/wiraedsus2019-web.pdf"
]
WEB_CSV_PATH = "/kaggle/input/rag-web-scrape-v4/detik_articles.csv"
LEGAL_JSON_PATH = "/kaggle/input/rag-uu/uu_34_tahun_2004.json"

pdf_documents = load_pdf_documents(PDF_SOURCES)
web_documents = load_web_articles(WEB_CSV_PATH)
legal_documents = load_legal_json(LEGAL_JSON_PATH)

all_documents = pdf_documents + web_documents + legal_documents

Loading PDF: /kaggle/input/rag-pdf-v2/Permohonan_4288_8190_Permohonan_redact.pdf
Loaded 24 pages from /kaggle/input/rag-pdf-v2/Permohonan_4288_8190_Permohonan_redact.pdf
Loading PDF: /kaggle/input/rag-pdf-v2/Petisi-Revisi-UU-TNI-.pdf
Loaded 11 pages from /kaggle/input/rag-pdf-v2/Petisi-Revisi-UU-TNI-.pdf
Loading PDF: /kaggle/input/rag-pdf-v2/naskah-akademik.pdf
Loaded 28 pages from /kaggle/input/rag-pdf-v2/naskah-akademik.pdf
Loading PDF: /kaggle/input/rag-pdf-v2/wiraedsus2019-web.pdf
Loaded 56 pages from /kaggle/input/rag-pdf-v2/wiraedsus2019-web.pdf
Loading web articles from: /kaggle/input/rag-web-scrape-v4/detik_articles.csv
Loaded 239 web articles
Loading legal JSON from: /kaggle/input/rag-uu/uu_34_tahun_2004.json
Loaded 78 legal articles from JSON


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  
    chunk_overlap=150,  
    length_function=len,
    separators=["\n\n", "\n", ". ", "! ", "? ", "; ", ", ", " "],  
    is_separator_regex=False
)

chunks = []
for doc in all_documents:
    if doc.metadata.get('type') == 'legal_article':
        chunks.append(doc)
    else:
        doc_chunks = text_splitter.split_documents([doc])
        chunks.extend(doc_chunks)

quality_chunks = []
for chunk in chunks:
    content = chunk.page_content.strip()
    if (len(content) > 50 and 
        len(content.split()) > 10 and
        not content.lower().startswith(('halaman', 'page', 'gambar', 'tabel'))):
        quality_chunks.append(chunk)

print(f"Total chunks created: {len(quality_chunks)}")
# print(f"Legal article chunks: {len([c for c in quality_chunks if c.metadata.get('type') == 'legal_article'])}")

Total chunks created: 1594


In [14]:
vectorstore = Chroma.from_documents(
    documents=quality_chunks,
    embedding=embeddings,
    persist_directory="./enhanced_chroma_db_v4",
    collection_metadata={"hnsw:space": "cosine"}
)

In [15]:
document_texts = [chunk.page_content for chunk in quality_chunks]
tokenized_docs = [doc.lower().split() for doc in document_texts]
bm25 = BM25Okapi(tokenized_docs)

In [16]:
def retrieve_documents_hybrid(query, k=8, alpha=0.7):
    semantic_results = vectorstore.similarity_search_with_score(query, k=k*2)
    
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    
    bm25_indices = np.argsort(bm25_scores)[::-1][:k*2]
    
    combined_results = {}
    
    for doc, score in semantic_results:
        doc_index = None
        for i, chunk in enumerate(quality_chunks):
            if chunk.page_content == doc.page_content:
                doc_index = i
                break
        
        if doc_index is not None:
            normalized_semantic = 1 / (1 + score)
            combined_results[doc_index] = {
                'doc': doc,
                'semantic_score': normalized_semantic,
                'bm25_score': 0
            }
    
    max_bm25_score = max(bm25_scores) if max(bm25_scores) > 0 else 1
    for idx in bm25_indices:
        normalized_bm25 = bm25_scores[idx] / max_bm25_score
        
        if idx in combined_results:
            combined_results[idx]['bm25_score'] = normalized_bm25
        else:
            combined_results[idx] = {
                'doc': quality_chunks[idx],
                'semantic_score': 0,
                'bm25_score': normalized_bm25
            }
    
    for idx in combined_results:
        semantic_score = combined_results[idx]['semantic_score']
        bm25_score = combined_results[idx]['bm25_score']
        combined_results[idx]['combined_score'] = alpha * semantic_score + (1 - alpha) * bm25_score
    
    sorted_results = sorted(combined_results.items(), 
                          key=lambda x: x[1]['combined_score'], 
                          reverse=True)
    
    return [result[1]['doc'] for result in sorted_results[:k]]

def rerank_documents(query, documents, top_k=6):
    if len(documents) <= top_k:
        return documents
    
    pairs = [[query, doc.page_content] for doc in documents]
    
    scores = cross_encoder.predict(pairs)
    
    doc_score_pairs = list(zip(documents, scores))
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
    
    return [doc for doc, score in doc_score_pairs[:top_k]]

def retrieve_documents(query, k=8):
    hybrid_results = retrieve_documents_hybrid(query, k=k*2)
    
    reranked_results = rerank_documents(query, hybrid_results, top_k=k)
    
    results = []
    seen_sources = set()
    
    for doc in reranked_results:
        if doc.metadata.get('type') == 'legal_article':
            source_key = f"pasal_{doc.metadata.get('pasal', 'unknown')}"
        else:
            source_key = f"{doc.metadata.get('source', '')}_{doc.metadata.get('page', 0)}"
        
        if source_key not in seen_sources:
            results.append(doc)
            seen_sources.add(source_key)
        elif len(results) < k//2:
            results.append(doc)
    
    return results[:k]


In [17]:
def generate_rag_response_with_lora(query, max_new_tokens=1000):
    retrieved_docs = retrieve_documents(query, k=6)
    
    context_parts = []
    for i, doc in enumerate(retrieved_docs, 1):
        if doc.metadata.get('type') == 'legal_article':
            source_info = f"Pasal {doc.metadata.get('pasal', 'Unknown')} UU"
        else:
            doc_type = doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown'))
            source_info = f"Sumber {i} ({doc_type})"
        
        context_parts.append(f"{source_info}:\n{doc.page_content}")
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""
### Peran:
Ahli Hukum Strategis (Indonesia).
### Konteks:
{context[:3000]}
### Struktur Jawaban:
1. Inti Jawaban:
    - Berikan jawaban langsung dan ringkas.
    - Sebutkan implikasi praktis utamanya.
2. Rincian Analisis:
    a. Isu Pokok: Identifikasi pertanyaan hukum spesifik.
    b. Aturan & Unsur: Uraikan aturan hukum (wajib sitasi pasal/UU TNI) dan unsur-unsurnya.
    c. Penerapan: Analisis logis bagaimana aturan berlaku pada fakta.
3. Disclaimer.
### Pertanyaan:
{query}
### Jawaban:
"""
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model_lora_lawbot.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id else tokenizer.pad_token_id
    )
    
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    if "### Jawaban:" in response:
        answer = response.split("### Jawaban:")[-1].strip()
    else:
        answer = response
    
    return answer, retrieved_docs

## Evaluation on RAG Metrics

In [18]:
def calculate_precision_recall(retrieved_docs, expected_contexts, expected_pasal=None, k_values=[3, 5, 8]):
    results = {}
    
    for k in k_values:
        retrieved_k = retrieved_docs[:k]
        
        relevant_docs = 0
        exact_matches = 0
        pasal_matches = 0
        
        for doc in retrieved_k:
            doc_content = doc.page_content.lower()
            doc_relevant = False
            doc_exact = False
            
            if doc.metadata.get('type') == 'legal_article' and expected_pasal:
                doc_pasal = doc.metadata.get('pasal', '')
                if str(doc_pasal) in [str(p) for p in expected_pasal]:
                    pasal_matches += 1
                    doc_relevant = True
                    doc_exact = True
            
            for expected in expected_contexts:
                expected_lower = expected.lower()
                
                if expected_lower in doc_content:
                    doc_exact = True
                    doc_relevant = True
                    break
                else:
                    expected_words = set(expected_lower.split())
                    doc_words = set(doc_content.split())
                    overlap = len(expected_words.intersection(doc_words))
                    if overlap >= len(expected_words) * 0.6:
                        doc_relevant = True
            
            if doc_relevant:
                relevant_docs += 1
                if doc_exact:
                    exact_matches += 1
        
        found_contexts = 0
        retrieved_full = " ".join([doc.page_content.lower() for doc in retrieved_k])
        
        for expected in expected_contexts:
            expected_lower = expected.lower()
            if expected_lower in retrieved_full:
                found_contexts += 1
            else:
                expected_words = set(expected_lower.split())
                retrieved_words = set(retrieved_full.split())
                overlap = len(expected_words.intersection(retrieved_words))
                if overlap >= len(expected_words) * 0.6:
                    found_contexts += 0.5
        
        precision_k = relevant_docs / k if k > 0 else 0
        recall_k = found_contexts / len(expected_contexts) if len(expected_contexts) > 0 else 0
        
        results[f"precision@{k}"] = precision_k
        results[f"recall@{k}"] = recall_k
        results[f"exact_matches@{k}"] = exact_matches
        results[f"pasal_matches@{k}"] = pasal_matches
    
    return results


In [19]:
ground_truth = [
    {
        "query": "Apa fungsi utama TNI menurut UU TNI?",
        "expected_contexts": [
            "fungsi TNI sebagai alat pertahanan negara",
            "TNI bertugas menjaga kedaulatan",
            "pertahanan dan keamanan negara", 
            "melindungi keutuhan wilayah NKRI",
            "mempertahankan kemerdekaan Indonesia",
            "menghadapi ancaman militer",
            "menjaga integritas teritorial"
        ],
        "expected_pasal": [4, 5, 6, 7]
    },
    {
        "query": "Apa perbedaan UU TNI lama dan revisi 2025?",
        "expected_contexts": [
            "revisi UU TNI 2025",
            "perubahan UU TNI",
            "UU TNI yang baru",
            "perbedaan UU lama dan baru",
            "pembaruan undang-undang TNI",
            "batas usia pensiun perwira",
            "perluasan peran OMSP",
            "jabatan sipil untuk militer aktif"
        ],
        "expected_pasal": None
    },
    {
        "query": "Bagaimana TNI berperan dalam operasi militer selain perang?",
        "expected_contexts": [
            "operasi militer selain perang",
            "OMSP TNI",
            "penanggulangan bencana alam",
            "bantuan kemanusiaan",
            "operasi non-perang",
            "misi perdamaian internasional",
            "kontra-terorisme",
            "pengamanan perbatasan",
            "search and rescue SAR"
        ],
        "expected_pasal": [7, 47]
    },
    {
        "query": "Apa kritik terhadap revisi UU TNI 2025?",
        "expected_contexts": [
            "kritik terhadap UU TNI",
            "penolakan revisi UU TNI",
            "kontroversi UU TNI 2025",
            "keberatan masyarakat sipil",
            "pro kontra revisi",
            "kekhawatiran dwifungsi TNI",
            "pelanggaran supremasi sipil",
            "transparansi pembahasan",
            "demokratisasi Indonesia"
        ],
        "expected_pasal": None
    },
    {
        "query": "Bagaimana hubungan TNI dengan pemerintah sipil?",
        "expected_contexts": [
            "TNI di bawah Presiden",
            "supremasi sipil atas militer",
            "hubungan sipil militer",
            "kontrol sipil terhadap TNI",
            "pemerintahan demokratis",
            "akuntabilitas TNI",
            "profesionalisme militer",
            "netralitas TNI dalam politik"
        ],
        "expected_pasal": [13, 14, 42, 43, 44]
    },
    {
        "query": "Apa itu dwifungsi TNI dan mengapa menjadi perdebatan?",
        "expected_contexts": [
            "dwifungsi TNI ABRI",
            "peran sosial politik militer",
            "dwifungsi era Orde Baru",
            "reformasi sektor militer",
            "perdebatan dwifungsi",
            "supremasi sipil vs militer",
            "demokrasi dan militer",
            "pembatasan peran politik TNI",
            "reformasi 1998"
        ],
        "expected_pasal": None
    },
    {
        "query": "Bagaimana proses pengangkatan perwira tinggi TNI?",
        "expected_contexts": [
            "pengangkatan perwira tinggi TNI",
            "mutasi dan promosi TNI",
            "jabatan militer strategis",
            "promosi perwira senior",
            "karier militer TNI",
            "persetujuan DPR",
            "evaluasi kinerja perwira",
            "sistem kepangkatan militer",
            "fit and proper test"
        ],
        "expected_pasal": [25, 26, 27, 28, 29, 30]
    },
    {
        "query": "Bagaimana struktur organisasi TNI?",
        "expected_contexts": [
            "organisasi TNI",
            "struktur komando TNI",
            "angkatan darat laut udara",
            "markas besar TNI",
            "panglima TNI",
            "kepala staf angkatan"
        ],
        "expected_pasal": [8, 9, 10, 11, 12]
    },
    {
        "query": "Bagaimana pengaturan anggaran dan keuangan TNI?",
        "expected_contexts": [
            "anggaran TNI",
            "pembiayaan pertahanan",
            "keuangan TNI",
            "APBN untuk TNI",
            "penggunaan anggaran",
            "pertanggungjawaban keuangan"
        ],
        "expected_pasal": [67, 68, 69, 70]
    }
]


In [20]:
def evaluate_rag():
    all_scores = []
    generated_answers = []
    
    for i, item in enumerate(ground_truth):
        query = item["query"]
        expected_contexts = item["expected_contexts"]
        expected_pasal = item.get("expected_pasal")
        
        print(f"\nEvaluating query {i+1}: {query}")
        
        answer, retrieved_docs = generate_rag_response_with_lora(query)
        generated_answers.append(answer)
        
        scores = calculate_precision_recall(retrieved_docs, expected_contexts, expected_pasal)
        all_scores.append(scores)
        
        legal_docs_found = [doc for doc in retrieved_docs if doc.metadata.get('type') == 'legal_article']
        pasal_found = [doc.metadata.get('pasal') for doc in legal_docs_found if doc.metadata.get('pasal')]
        
        print(f"Answer preview: {answer[:100]}...")
        print(f"Legal docs found: {len(legal_docs_found)}, Pasal found: {pasal_found}")
        
        for k in [3, 5, 8]:
            precision = scores[f'precision@{k}']
            recall = scores[f'recall@{k}']
            exact = scores[f'exact_matches@{k}']
            pasal = scores[f'pasal_matches@{k}']
            
            print(f"  k={k}: P={precision:.3f}, R={recall:.3f}, Exact={exact}, Pasal={pasal}")
    
    avg_scores = {}
    for k in [3, 5, 8]:
        avg_scores[f"avg_precision@{k}"] = np.mean([scores[f"precision@{k}"] for scores in all_scores])
        avg_scores[f"avg_recall@{k}"] = np.mean([scores[f"recall@{k}"] for scores in all_scores])
        avg_scores[f"avg_exact_matches@{k}"] = np.mean([scores[f"exact_matches@{k}"] for scores in all_scores])
        avg_scores[f"avg_pasal_matches@{k}"] = np.mean([scores[f"pasal_matches@{k}"] for scores in all_scores])
    
    print("\n" + "="*60)
    print("RAG EVALUATION RESULTS")
    print("="*60)
    
    for k in [3, 5, 8]:
        print(f"\nAt k={k}:")
        print(f"  Precision: {avg_scores[f'avg_precision@{k}']:.3f}")
        print(f"  Recall: {avg_scores[f'avg_recall@{k}']:.3f}")
        print(f"  Avg Exact Matches: {avg_scores[f'avg_exact_matches@{k}']:.1f}")
        print(f"  Avg Pasal Matches: {avg_scores[f'avg_pasal_matches@{k}']:.1f}")
    
    return avg_scores, generated_answers

In [21]:
avg_scores, generated_answers = evaluate_rag()

summary_data = {
    "k": [3, 5, 8],
    "Precision": [avg_scores[f"avg_precision@{k}"] for k in [3, 5, 8]],
    "Recall": [avg_scores[f"avg_recall@{k}"] for k in [3, 5, 8]],
    "Avg Exact Matches": [avg_scores[f"avg_exact_matches@{k}"] for k in [3, 5, 8]],
    "Avg Pasal Matches": [avg_scores[f"avg_pasal_matches@{k}"] for k in [3, 5, 8]],
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("rag_evaluation_metrics_summary.csv", index=False)

answers_df = pd.DataFrame({
    "query": [item["query"] for item in ground_truth],
    "generated_answer": generated_answers
})
answers_df.to_csv("rag_generated_answers.csv", index=False)


display(FileLink("rag_evaluation_metrics_summary.csv"))


Evaluating query 1: Apa fungsi utama TNI menurut UU TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: Fungsi utama TNI adalah sebagai alat pertahanan negara untuk menghadapi ancaman militer dan bersenja...
Legal docs found: 3, Pasal found: [12, 5, 6]
  k=3: P=0.667, R=0.214, Exact=0, Pasal=0
  k=5: P=0.800, R=0.286, Exact=2, Pasal=2
  k=8: P=0.625, R=0.357, Exact=3, Pasal=2

Evaluating query 2: Apa perbedaan UU TNI lama dan revisi 2025?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: Perbedaan utama adalah perluasan peran TNI di sektor sipil dan perpanjangan masa dinas prajurit. Imp...
Legal docs found: 0, Pasal found: []
  k=3: P=1.000, R=0.312, Exact=0, Pasal=0
  k=5: P=1.000, R=0.312, Exact=0, Pasal=0
  k=8: P=0.625, R=0.312, Exact=0, Pasal=0

Evaluating query 3: Bagaimana TNI berperan dalam operasi militer selain perang?


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: 1. Implikasi: Hierarki TNI mencerminkan komponen pertahanan negara.
2. Analisis: Pasal 12 mengatur s...
Legal docs found: 5, Pasal found: [11, 12, 6, 5, 45]
  k=3: P=0.667, R=0.750, Exact=2, Pasal=2
  k=5: P=0.600, R=0.750, Exact=3, Pasal=2
  k=8: P=0.375, R=0.750, Exact=3, Pasal=2

Evaluating query 9: Bagaimana pengaturan anggaran dan keuangan TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer preview: 1. Implikasi Praktis: Sistem pengawasan dan pemeriksaan anggaran TNI oleh BPK RI. 2. Isu & Analisis:...
Legal docs found: 3, Pasal found: [69, 66, 67]
  k=3: P=0.667, R=0.167, Exact=1, Pasal=1
  k=5: P=0.800, R=0.417, Exact=2, Pasal=2
  k=8: P=0.500, R=0.417, Exact=2, Pasal=2

RAG EVALUATION RESULTS

At k=3:
  Precision: 0.815
  Recall: 0.286
  Avg Exact Matches: 0.7
  Avg Pasal Matches: 0.3

At k=5:
  Precision: 0.711
  Recall: 0.341
  Avg Exact Matches: 1.1
  Avg Pasal Matches: 0.7

At k=8:
  Precision: 0.458
  Recall: 0.349
  Avg Exact Matches: 1.2
  Avg Pasal Matches: 0.7


/kaggle/working/rag_evaluation_metrics_summary.csv

## Evaluation with LLM Judge

In [22]:
import pandas as pd
from datasets import Dataset, concatenate_datasets
df = pd.read_csv("/kaggle/input/dataset-with-format-answer-v2/dataset-formated-fix.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (2623, 3)


,Question,Answer,pasal
0,Apa yang dimaksud dengan 'keadaan darurat mili...,1. Inti Jawaban:\n- 'Keadaan darurat militer' ...,1
1,Bagaimana proses pemanggilan kembali prajurit ...,1. Inti Jawaban:\n- Proses pemanggilan kembali...,1
2,Apakah prajurit yang telah selesai masa dinas ...,"1. Inti Jawaban:\n- Ya, prajurit yang telah se...",1
3,Apa yang diatur dalam Pasal 60 ayat 1 tentang ...,Pasal 60 ayat 1 mengatur bahwa prajurit sukare...,1
4,Apa yang dimaksud dengan Negara menurut UU No ...,1. Inti Jawaban:\n- Negara adalah Negara Kesat...,1


In [23]:
TEST_PROPORTION = 0.1
SEED = 123

# Split dataset (same as original)
unique_pasals = df['pasal'].unique()
train_splits = {}
test_splits = {}

for pasal_value in unique_pasals:
    group_df = df[df['pasal'] == pasal_value]
    dataset_group = Dataset.from_pandas(group_df)
    
    if len(dataset_group) > 1:
        split = dataset_group.train_test_split(test_size=TEST_PROPORTION, seed=SEED)
        train_splits[pasal_value] = split['train']
        test_splits[pasal_value] = split['test']
    else:
        train_splits[pasal_value] = dataset_group
        test_splits[pasal_value] = None

final_train_list = [ds for ds in train_splits.values() if ds is not None]
final_test_list = [ds for ds in test_splits.values() if ds is not None]

final_eval_dataset_raw = concatenate_datasets(final_test_list).shuffle(seed=SEED)

In [24]:
final_eval_dataset_raw

Dataset({
    features: ['Question', 'Answer', 'pasal', '__index_level_0__'],
    num_rows: 291
})

In [25]:
import torch
from sentence_transformers import SentenceTransformer, util

device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_embedding_model = SentenceTransformer(embedding_model_name, device=device)

In [26]:
# Fungsi untuk menghitung relevansi dokumen yang di-retrieve
def calculate_retrieval_relevance(query, retrieved_texts, embedding_model):
    """
    Menghitung relevansi dokumen yang di-retrieve terhadap query
    """
    if not retrieved_texts:
        return 0.0
    
    # Gabungkan semua teks yang di-retrieve
    combined_retrieved = "\n".join(retrieved_texts.split("\n\n---\n\n"))
    
    # Hitung similarity antara query dan dokumen yang di-retrieve
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)
    retrieved_embedding = embedding_model.encode(combined_retrieved, convert_to_tensor=True)
    
    similarity = util.pytorch_cos_sim(query_embedding, retrieved_embedding).item()
    return similarity

In [27]:
# Ambil 5 baris pertama dari dataset evaluasi
test_samples = final_eval_dataset_raw.select(range(5))

# Modifikasi bagian evaluasi untuk 5 baris pertama
predictions = []
references = []
questions = []
retrieved_contents = []

# Iterate through 5 test samples
for ex in tqdm(test_samples, desc="Testing 5 samples"):
    question = ex["Question"]
    reference_answer = ex["Answer"]
    
    # Generate RAG-enhanced answer
    answer, retrieved_docs = generate_rag_response_with_lora(question)
    
    # Simpan konten dokumen yang di-retrieve
    retrieved_texts = []
    for doc in retrieved_docs:
        content = doc.page_content
        source = doc.metadata.get('source', 'unknown')
        doc_type = doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown'))
        retrieved_text = f"[{doc_type}] {content[:500]}..." 
        retrieved_texts.append(retrieved_text)
    
    predictions.append(answer)
    references.append(reference_answer)
    questions.append(question)
    retrieved_contents.append("\n\n---\n\n".join(retrieved_texts))

# Hitung relevansi retrieval
retrieval_scores = []
for q, retrieved in zip(questions, retrieved_contents):
    score = calculate_retrieval_relevance(q, retrieved, eval_embedding_model)
    retrieval_scores.append(score)

# Buat DataFrame untuk 5 sampel
df_test_results = pd.DataFrame({
    'Questions': questions,
    'Reference Answer': references,
    'Generated Answer': predictions,
    'Retrieved Contents': retrieved_contents,
    'Retrieval Relevance Score': retrieval_scores
})

df_test_results.to_csv('result_generate_with_retrieval_test.csv', index=False)

# Tampilkan hasil dengan format yang lebih baik
pd.set_option('display.max_colwidth', 100)
print("\n=== Hasil Test 5 Sampel ===")
display(df_test_results[['Questions', 'Retrieval Relevance Score']])

print("\n=== Contoh Lengkap 1 Sampel ===")
sample_idx = 2
print(f"\nQuestion: {df_test_results['Questions'].iloc[sample_idx]}")
print(f"\nReference Answer:\n{df_test_results['Reference Answer'].iloc[sample_idx]}")
print(f"\nGenerated Answer:\n{df_test_results['Generated Answer'].iloc[sample_idx]}")
print(f"\nRetrieved Contents:\n{df_test_results['Retrieved Contents'].iloc[sample_idx]}")
print(f"\nRetrieval Relevance Score: {df_test_results['Retrieval Relevance Score'].iloc[sample_idx]:.4f}")

# Hitung rata-rata skor untuk 5 sampel
avg_score = np.mean(retrieval_scores)
print(f"\n=== Rangkuman ===")
print(f"Average Retrieval Relevance Score (5 samples): {avg_score:.4f}")

Testing 5 samples:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Testing 5 samples:  20%|██        | 1/5 [00:02<00:11,  2.86s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Testing 5 samples:  40%|████      | 2/5 [00:09<00:16,  5.35s/it]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


=== Hasil Test 5 Sampel ===


,Questions,Retrieval Relevance Score
0,Siapa yang memimpin TNI menurut Pasal 13 UU No. 34 Tahun 2004?,0.712216
1,Bagaimana Pasal 37 ayat (2) mengatur kerahasiaan militer bagi mantan prajurit?,0.686463
2,Apa syarat untuk menjadi Panglima TNI berdasarkan Pasal 13?,0.705582
3,Dalam kondisi apa Pasal 60 UU TNI dapat diterapkan?,0.756548
4,Apakah BPK RI memiliki kewenangan penuh dalam mengaudit anggaran TNI tanpa campur tangan pihak l...,0.642023



=== Contoh Lengkap 1 Sampel ===

Question: Apa syarat untuk menjadi Panglima TNI berdasarkan Pasal 13?

Reference Answer:
1. Inti Jawaban:
- Harus Perwira Tinggi aktif dari salah satu Angkatan.
- Implikasi praktis: Memastikan Panglima memiliki pengalaman militer yang memadai.

2. Rincian Analisis:
a. Isu Pokok: Kualifikasi Panglima.
b. Aturan & Unsur: Pasal 13 ayat (4) menyebutkan syarat pernah menjabat sebagai Kepala Staf Angkatan.
c. Penerapan: Rotasi kepemimpinan antar Angkatan dimungkinkan.

3. Disclaimer: Persyaratan tambahan mungkin diatur dalam keputusan Presiden.

Generated Answer:
Syarat menjadi Panglima TNI meliputi: (1) diangkat oleh Presiden setelah persetujuan DPR, (2) harus memenuhi kepentingan organisasi TNI, dan (3) jabatan dapat diisi oleh Perwira Tinggi aktif dari tiap angkatan. Implikasi praktisnya adalah proses pengangkatan Panglima harus melalui mekanisme demokratis dengan partisipasi legislatif.

Retrieved Contents:
[undang_undang] Pasal 13 Isi: (1) TNI dipimpin 

In [28]:
from IPython.display import FileLink, display
display(FileLink("result_generate_with_retrieval_test.csv"))

/kaggle/working/result_generate_with_retrieval_test.csv

In [ ]:
# Modifikasi bagian evaluasi untuk menyimpan teks dokumen yang di-retrieve
predictions = []
references = []
questions = []
retrieved_contents = []  # Ganti dari retrieved_sources ke retrieved_contents

# Iterate through evaluation dataset with RAG
for ex in tqdm(final_eval_dataset_raw, desc="Mengevaluasi Model dengan RAG"):
    question = ex["Question"]
    reference_answer = ex["Answer"]
    
    # Generate RAG-enhanced answer
    answer, retrieved_docs = generate_rag_response_with_lora(question)
    
    # Simpan konten dokumen yang di-retrieve (bukan hanya sumbernya)
    retrieved_texts = []
    for doc in retrieved_docs:
        # Ambil konten dokumen dan metadata penting
        content = doc.page_content
        source = doc.metadata.get('source', 'unknown')
        doc_type = doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown'))
        
        # Format teks yang akan disimpan
        retrieved_text = f"[{doc_type}] {content[:500]}..."  # Batasi panjang teks
        retrieved_texts.append(retrieved_text)
    
    # Store results
    predictions.append(answer)
    references.append(reference_answer)
    questions.append(question)
    retrieved_contents.append("\n\n---\n\n".join(retrieved_texts))  # Gabungkan dengan pemisah



# Hitung relevansi retrieval untuk setiap query
retrieval_scores = []
for q, retrieved in zip(questions, retrieved_contents):
    score = calculate_retrieval_relevance(q, retrieved, eval_embedding_model)
    retrieval_scores.append(score)

# Save results to CSV
df_results = pd.DataFrame({
    'Questions': questions,
    'Reference Answer': references,
    'Generated Answer': predictions,
    'Retrieved Contents': retrieved_contents,
    'Retrieval Relevance Score': retrieval_scores
})

# Hitung metrik tambahan untuk evaluasi retrieval
avg_retrieval_score = np.mean(retrieval_scores)
print(f"\n--- Hasil Evaluasi Retrieval ---")
print(f"Average Retrieval Relevance Score: {avg_retrieval_score:.4f}")
print("Skor 1.0 berarti sangat relevan, 0.0 berarti tidak relevan")

df_results.to_csv('result_generate.csv', index=False)

Mengevaluasi Model dengan RAG:   0%|          | 0/291 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Mengevaluasi Model dengan RAG:   0%|          | 1/291 [00:02<12:51,  2.66s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Mengevaluasi Model dengan RAG:   1%|          | 2/291 [00:09<24:20,  5.05s/it]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
import pandas as pd
import torch
from tqdm.auto import tqdm
import numpy as np
import subprocess
import sys

# --- 1. Instalasi dan Setup Library Tambahan ---

# Pastikan library yang dibutuhkan untuk evaluasi semantik terinstal
print("Memeriksa dan menginstal library yang dibutuhkan...")
try:
    import nltk
    from sentence_transformers import SentenceTransformer, util
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError:
    print("Menginstal 'sentence-transformers', 'nltk', dan 'scikit-learn'...")
    subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers", "nltk", "scikit-learn"], check=True)
    from sentence_transformers import SentenceTransformer, util
    from sklearn.metrics.pairwise import cosine_similarity
    import nltk

# Download tokenizer 'punkt' dari NLTK untuk memecah kalimat
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    print("Mengunduh NLTK 'punkt' tokenizer...")
    nltk.download('punkt')

# --- 2. Muat Model Embedding ---

# Memuat model embedding multilingual yang kuat.
# Qwen/Qwen2-57B-A14B-instruct-embedding adalah model yang sangat besar. 
# Untuk efisiensi, kita gunakan model yang lebih ringan namun tetap powerful untuk multilingual.
# 'paraphrase-multilingual-mpnet-base-v2' adalah pilihan yang sangat baik.
embedding_model_name = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
print(f"Memuat model embedding ({embedding_model_name})...")
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print("Model embedding berhasil dimuat.")
print("-" * 50)


# --- 3. Fungsi untuk Menghitung Skor Semantik ---

def calculate_semantic_scores(reference, prediction, model, threshold=0.85):
    """
    Menghitung precision, recall, dan F1-score berdasarkan kesamaan semantik.
    """
    # Memecah jawaban menjadi kalimat-kalimat
    ref_sentences = nltk.sent_tokenize(reference)
    pred_sentences = nltk.sent_tokenize(prediction)

    if not pred_sentences or not ref_sentences:
        return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

    # Generate embeddings untuk setiap kalimat
    ref_embeddings = model.encode(ref_sentences, convert_to_tensor=True)
    pred_embeddings = model.encode(pred_sentences, convert_to_tensor=True)

    # Hitung cosine similarity
    cosine_scores = util.cos_sim(pred_embeddings, ref_embeddings)

    # Hitung Precision
    pred_matches = 0
    for i in range(len(pred_sentences)):
        if cosine_scores.shape[1] > 0 and torch.max(cosine_scores[i]) > threshold:
            pred_matches += 1
    precision = pred_matches / len(pred_sentences)

    # Hitung Recall
    ref_matches = 0
    for j in range(len(ref_sentences)):
        if cosine_scores.shape[0] > 0 and torch.max(cosine_scores[:, j]) > threshold:
            ref_matches += 1
    recall = ref_matches / len(ref_sentences)
    
    # Hitung F1 Score
    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
        
    return {'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}


# --- 4. Terapkan Fungsi Evaluasi ke DataFrame Hasil ---

# Asumsikan df_results sudah ada dari kode sebelumnya dan memiliki kolom
# 'Reference Answer' dan 'Generate Answer'.
# Contoh:
# df_results = pd.read_csv('result_generate.csv')

scores = []
# Iterasi melalui setiap baris di DataFrame hasil
for index, row in tqdm(df_results.iterrows(), total=df_results.shape[0], desc="Menghitung Skor Semantik"):
    
    # --- MODIFIKASI NAMA KOLOM DI SINI ---
    # Menggunakan nama kolom 'Reference Answer' dan 'Generate Answer'
    # Menggunakan str() untuk memastikan tipe data string dan menangani NaN
    ref = str(row['Reference Answer'])
    pred = str(row['Generated Answer'])
    
    # Hitung skor untuk baris saat ini
    score = calculate_semantic_scores(ref, pred, embedding_model)
    scores.append(score)

# Tambahkan skor ke DataFrame
df_scores = pd.DataFrame(scores)
df_results_with_scores = pd.concat([df_results, df_scores], axis=1)


# --- 5. Tampilkan Hasil Akhir ---

# Hitung rata-rata skor
avg_precision = df_results_with_scores['precision'].mean()
avg_recall = df_results_with_scores['recall'].mean()
avg_f1 = df_results_with_scores['f1'].mean()

print("\n--- Hasil Evaluasi Semantik (Rata-rata) ---")
print(f"Average Precision : {avg_precision * 100:.2f}%")
print(f"Average Recall    : {avg_recall * 100:.2f}%")
print(f"Average F1-Score  : {avg_f1 * 100:.2f}%")
print("-" * 50)


# Simpan hasil akhir yang sudah berisi skor semantik
output_scored_csv_path = 'result_generate_with_scores.csv'
df_results_with_scores.to_csv(output_scored_csv_path, index=False)
print(f"\nHasil evaluasi lengkap dengan skor berhasil disimpan ke: {output_scored_csv_path}")

Memeriksa dan menginstal library yang dibutuhkan...
Memuat model embedding (sentence-transformers/paraphrase-multilingual-mpnet-base-v2)...
Model embedding berhasil dimuat.
--------------------------------------------------


Menghitung Skor Semantik:   0%|          | 0/291 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Hasil Evaluasi Semantik (Rata-rata) ---
Average Precision : 35.26%
Average Recall    : 24.70%
Average F1-Score  : 27.43%
--------------------------------------------------

Hasil evaluasi lengkap dengan skor berhasil disimpan ke: result_generate_with_scores.csv


In [36]:
from IPython.display import FileLink, display
print("\nUnduh file hasil di bawah ini:")
display(FileLink("result_generate.csv"))
display(FileLink("result_generate_with_scores.csv"))


Unduh file hasil di bawah ini:


/kaggle/working/result_generate.csv

/kaggle/working/result_generate_with_scores.csv

In [ ]:
# Test RAG system
print("\n" + "="*70)
print("TESTING RAG SYSTEM")
print("="*70)

test_queries = [
    "Apa saja tugas pokok TNI menurut UU TNI?",
    "Bagaimana dampak revisi UU TNI 2025 terhadap peran TNI?",
    "Mengapa terjadi penolakan terhadap revisi UU TNI?",
    "Bagaimana aksi demo UU TNI di Surabaya berjalan?",
    "Bagaimana mekanisme pertanggungjawaban TNI kepada rakyat?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    answer, retrieved_docs = generate_rag_response_with_lora(query)
    print(f"RAG Answer: {answer}")
    print(f"Retrieved {len(retrieved_docs)} relevant documents")
    
    # Show source diversity
    source_types = [doc.metadata.get('doc_type', doc.metadata.get('type', 'unknown')) for doc in retrieved_docs]
    print(f"Source types: {', '.join(set(source_types))}")
    print("-" * 50)


TESTING RAG SYSTEM

Query: Apa saja tugas pokok TNI menurut UU TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: Tugas pokok TNI mencakup: 1. Memimpin TNI; 2. Melaksanakan kebijakan pertahanan negara; 3. Mengembangkan doktrin TNI; 4. Menyelenggarakan strategi militer dan operasi militer; 5. Memberikan pertimbangan kepada Menteri Pertahanan dalam kebijakan pertahanan negara. Implikasi praktisnya adalah TNI harus menjalankan fungsi pertahanan secara efektif dan efisien.
Retrieved 6 relevant documents
Source types: web_article, undang_undang, analysis_report
--------------------------------------------------

Query: Bagaimana dampak revisi UU TNI 2025 terhadap peran TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: 1. Inti Jawaban:
    - Dampaknya antara lain perluasan penempatan TNI di kementerian dan lembaga, serta perpanjangan masa pensiun prajurit.
2. Rincian Analisis:
    a. Isu Pokok: Implikasi revisi UU TNI.
    b. Aturan & Unsur: Pasal 8 ayat (2) huruf f UU No. 3/2025 menyebutkan perubahan terkait peran TNI.
    c. Penerapan: Revisi ini dapat meningkatkan peran TNI dalam sektor sipil dan memperpanjang masa dinas prajurit.
3. Disclaimer: Analisis ini bersifat umum dan tidak menggantikan konsultasi hukum.
Retrieved 4 relevant documents
Source types: legal_request, web_article
--------------------------------------------------

Query: Mengapa terjadi penolakan terhadap revisi UU TNI?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: 1. Inti Jawaban:
    - Penolakan disebabkan oleh ketidaktransparanan proses dan kekhawatiran atas potensi restorasi dwifungsi TNI.
2. Rincian Analisis:
    a. Isu Pokok: Alasan penolakan revisi UU TNI.
    b. Aturan & Unsur: Pasal 7 ayat 2 menjadi fokus kritik karena dianggap ambigu.
    c. Penerapan: Kritik bersumber dari berbagai kelompok termasuk mahasiswa dan akademisi.

```mermaid
flowchart TD
    A[Pembahasan Revisi] --> B[Kritik Transparansi]
    B --> C[Penolakan Massa]
    C --> D[Mekanisme Akuntabilitas]
```
Retrieved 3 relevant documents
Source types: web_article
--------------------------------------------------

Query: Bagaimana aksi demo UU TNI di Surabaya berjalan?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

RAG Answer: 1. Inti Jawaban:
    - Demo berlangsung damai dengan tuntutan 8 poin penolakan.
2. Rincian Analisis:
    a. Isu Pokok: Proses demonstrasi.
    b. Aturan & Unsur: Pasal 28 UU No. 34/2004 tentang hak berserikat.
    c. Implikasi: Demonstrasi sebagai bentuk kritik konstitusional.
3. Disclaimer: Data dari sumber media.
Retrieved 3 relevant documents
Source types: web_article
--------------------------------------------------

Query: Bagaimana mekanisme pertanggungjawaban TNI kepada rakyat?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]